In [ ]:
import pandas as pd

# --- 1. Load both files ---
# For demonstration, creating dummy dataframes. Replace with actual file loading:
# ledger_df = pd.read_csv('ledger.csv')
# gateway_df = pd.read_csv('gateway.csv')

# Dummy Data for demonstration
data_ledger = {
    'transaction_id': ['T001', 'T002', 'T003', 'T004', 'T005', 'T006'],
    'amount': [100.00, 200.00, 150.00, 300.00, 50.00, 250.00],
    'status': ['completed', 'pending', 'completed', 'completed', 'failed', 'completed'],
    'date': ['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05', '2023-01-06']
}
ledger_df = pd.DataFrame(data_ledger)

data_gateway = {
    'transaction_id': ['T001', 'T002', 'T003', 'T005', 'T007', 'T006'],
    'amount': [100.00, 200.00, 155.00, 50.00, 400.00, 250.00],
    'status': ['completed', 'completed', 'completed', 'failed', 'completed', 'pending'],
    'date': ['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-05', '2023-01-07', '2023-01-06']
}
gateway_df = pd.DataFrame(data_gateway)

print("--- Data Loaded ---")
print("Ledger Data:\n", ledger_df.head())
print("\nGateway Data:\n", gateway_df.head())

# --- 2. Check duplicates and nulls ---
print("\n--- Duplicate and Null Checks ---")

# Ledger Duplicates
ledger_duplicates = ledger_df[ledger_df.duplicated(subset='transaction_id', keep=False)]
print(f"\nLedger duplicate transaction_ids: {len(ledger_duplicates)}")
if not ledger_duplicates.empty:
    print(ledger_duplicates)

# Gateway Duplicates
gateway_duplicates = gateway_df[gateway_df.duplicated(subset='transaction_id', keep=False)]
print(f"\nGateway duplicate transaction_ids: {len(gateway_duplicates)}")
if not gateway_duplicates.empty:
    print(gateway_duplicates)

# Ledger Nulls
ledger_nulls = ledger_df.isnull().sum()
print("\nLedger Nulls per column:\n", ledger_nulls[ledger_nulls > 0])

# Gateway Nulls
gateway_nulls = gateway_df.isnull().sum()
print("\nGateway Nulls per column:\n", gateway_nulls[gateway_nulls > 0])

# --- Prepare for reconciliation by setting transaction_id as index ---
ledger_indexed = ledger_df.set_index('transaction_id')
gateway_indexed = gateway_df.set_index('transaction_id')

# --- 3. Identify records missing in gateway ---
missing_in_gateway = ledger_indexed[~ledger_indexed.index.isin(gateway_indexed.index)]
print("\n--- Records Missing in Gateway ---")
print(f"Number of records missing in Gateway: {len(missing_in_gateway)}")
if not missing_in_gateway.empty:
    print(missing_in_gateway)

# --- 4. Identify records missing in ledger ---
missing_in_ledger = gateway_indexed[~gateway_indexed.index.isin(ledger_indexed.index)]
print("\n--- Records Missing in Ledger ---")
print(f"Number of records missing in Ledger: {len(missing_in_ledger)}")
if not missing_in_ledger.empty:
    print(missing_in_ledger)

# --- Common records for detailed comparison ---
common_transactions = pd.merge(
    ledger_indexed,
    gateway_indexed,
    left_index=True,
    right_index=True,
    suffixes=('_ledger', '_gateway')
)

# --- 5. Identify amount mismatches ---
amount_mismatches = common_transactions[
    common_transactions['amount_ledger'] != common_transactions['amount_gateway']
]
print("\n--- Amount Mismatches ---")
print(f"Number of amount mismatches: {len(amount_mismatches)}")
if not amount_mismatches.empty:
    print(amount_mismatches[['amount_ledger', 'amount_gateway']])

# --- 6. Identify status mismatches ---
status_mismatches = common_transactions[
    common_transactions['status_ledger'] != common_transactions['status_gateway']
]
print("\n--- Status Mismatches ---")
print(f"Number of status mismatches: {len(status_mismatches)}")
if not status_mismatches.empty:
    print(status_mismatches[['status_ledger', 'status_gateway']])

# --- 7. Build a final reconciliation report ---
reconciliation_report = pd.DataFrame(columns=[
    'transaction_id', 'discrepancy_type', 'ledger_value', 'gateway_value'
])

# Helper to add to report
def add_to_report(report_df, transaction_id, disc_type, ledger_val, gateway_val):
    return pd.concat([
        report_df,
        pd.DataFrame([{
            'transaction_id': transaction_id,
            'discrepancy_type': disc_type,
            'ledger_value': ledger_val,
            'gateway_value': gateway_val
        }])
    ], ignore_index=True)

for idx, row in missing_in_gateway.iterrows():
    reconciliation_report = add_to_report(
        reconciliation_report, idx, 'Missing in Gateway', row.to_dict(), 'N/A'
    )

for idx, row in missing_in_ledger.iterrows():
    reconciliation_report = add_to_report(
        reconciliation_report, idx, 'Missing in Ledger', 'N/A', row.to_dict()
    )

for idx, row in amount_mismatches.iterrows():
    reconciliation_report = add_to_report(
        reconciliation_report, idx, 'Amount Mismatch',
        row['amount_ledger'], row['amount_gateway']
    )

for idx, row in status_mismatches.iterrows():
    reconciliation_report = add_to_report(
        reconciliation_report, idx, 'Status Mismatch',
        row['status_ledger'], row['status_gateway']
    )

print("\n--- Final Reconciliation Report ---")
if not reconciliation_report.empty:
    print(reconciliation_report)
else:
    print("No discrepancies found. All records reconciled.")

# --- 8. Generate summary metrics ---
print("\n--- Reconciliation Summary Metrics ---")
print(f"Total Ledger Records: {len(ledger_df)}")
print(f"Total Gateway Records: {len(gateway_df)}")
print(f"Duplicate Transaction IDs in Ledger: {len(ledger_duplicates)}")
print(f"Duplicate Transaction IDs in Gateway: {len(gateway_duplicates)}")
print(f"Records Missing in Gateway: {len(missing_in_gateway)}")
print(f"Records Missing in Ledger: {len(missing_in_ledger)}")
print(f"Amount Mismatches: {len(amount_mismatches)}")
print(f"Status Mismatches: {len(status_mismatches)}")
print(f"Total Discrepancies Found: {len(reconciliation_report)}")

# Optional: Save the report
# reconciliation_report.to_csv('reconciliation_report.csv', index=False)
# print("\nReconciliation report saved to 'reconciliation_report.csv'")

--- Data Loaded ---
Ledger Data:
   transaction_id  amount     status        date
0           T001   100.0  completed  2023-01-01
1           T002   200.0    pending  2023-01-02
2           T003   150.0  completed  2023-01-03
3           T004   300.0  completed  2023-01-04
4           T005    50.0     failed  2023-01-05

Gateway Data:
   transaction_id  amount     status        date
0           T001   100.0  completed  2023-01-01
1           T002   200.0  completed  2023-01-02
2           T003   155.0  completed  2023-01-03
3           T005    50.0     failed  2023-01-05
4           T007   400.0  completed  2023-01-07

--- Duplicate and Null Checks ---

Ledger duplicate transaction_ids: 0

Gateway duplicate transaction_ids: 0

Ledger Nulls per column:
 Series([], dtype: int64)

Gateway Nulls per column:
 Series([], dtype: int64)

--- Records Missing in Gateway ---
Number of records missing in Gateway: 1
                amount     status        date
transaction_id                       

In [ ]:
import json

# Load the nested JSON file
with open('/content/api_response_sample.json', 'r') as f:
    nested_json_data = json.load(f)

print('Loaded JSON data structure type:', type(nested_json_data))
# Display a part of the loaded JSON to understand its structure
# For simplicity, if it's a list, show the first item; otherwise, show the whole dict
if isinstance(nested_json_data, list):
    print('First item of JSON data:')
    print(json.dumps(nested_json_data[0], indent=2))
else:
    print('JSON data (first few keys):')
    print(json.dumps(dict(list(nested_json_data.items())[:3]), indent=2))


Loaded JSON data structure type: <class 'dict'>
JSON data (first few keys):
{
  "generated_at": "2026-03-07T10:00:00Z",
  "source": "QuickPay Settlement API",
  "batches": [
    {
      "batch_id": "B001",
      "merchant": {
        "merchant_id": "M001",
        "merchant_name": "Alpha Mart",
        "region": "APAC"
      },
      "settlements": [
        {
          "settlement_id": "S001",
          "amount_usd": 1520.5,
          "status": "settled",
          "processed_at": "2026-03-07T08:10:00Z",
          "bank": {
            "name": "Bank A",
            "country": "IN"
          }
        },
        {
          "settlement_id": "S002",
          "amount_usd": 980.0,
          "status": "pending",
          "processed_at": "2026-03-07T08:45:00Z",
          "bank": {
            "name": "Bank A",
            "country": "IN"
          }
        },
        {
          "settlement_id": "S003",
          "amount_usd": 640.0,
          "status": "settled",
          "processed_at

In [ ]:
import pandas as pd

# Flatten the JSON data
# The main records to flatten are the 'settlements' nested within 'batches'.
# We also want to include metadata from the top level (generated_at, source) and
# from the 'batches' level (batch_id, merchant details).
normalized_df = pd.json_normalize(
    nested_json_data,
    record_path=['batches', 'settlements'],
    meta=[
        'generated_at',
        'source',
        ['batches', 'batch_id'],
        ['batches', 'merchant', 'merchant_id'],
        ['batches', 'merchant', 'merchant_name'],
        ['batches', 'merchant', 'region']
    ]
)

print('Normalized DataFrame shape:', normalized_df.shape)
display(normalized_df.head())

Normalized DataFrame shape: (1, 3)


,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


Now, let's clean the column names by converting them to snake_case and removing special characters for easier handling.

In [ ]:
import re

def clean_column_names(df):
    cols = df.columns
    new_cols = []
    for col in cols:
        # Replace spaces and dots with underscores, convert to lowercase
        new_col = re.sub(r'[\s\.]', '_', col)
        # Remove any characters that are not alphanumeric or underscore
        new_col = re.sub(r'[^a-zA-Z0-9_]', '', new_col)
        # Convert to snake_case (handle CamelCase/PascalCase if present, though json_normalize often creates flat names)
        new_col = re.sub(r'(?<!^)(?=[A-Z])', '_', new_col).lower()
        new_cols.append(new_col)
    df.columns = new_cols
    return df

normalized_df = clean_column_names(normalized_df)
print('Cleaned column names:', normalized_df.columns.tolist())
display(normalized_df.head())


Cleaned column names: ['generated_at', 'source', 'batches']


,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


In [ ]:
# Convert date/time fields
for col in normalized_df.columns:
    # Heuristic to identify potential date/time columns
    # Can be extended based on actual column names/data types if known
    if 'date' in col or 'time' in col or 'created' in col or 'updated' in col:
        # Attempt to convert to datetime, coercing errors to NaT
        normalized_df[col] = pd.to_datetime(normalized_df[col], errors='coerce')
        # Drop column if all values became NaT after conversion (wasn't a date column)
        if normalized_df[col].isnull().all() and 'object' in str(normalized_df[col].dtype):
            print(f"Warning: Column '{col}' could not be converted to datetime and will remain its original type or dropped if all NaT.")
        else:
            print(f"Converted column '{col}' to datetime.")

print('\nDataFrame info after datetime conversion:')
normalized_df.info()
display(normalized_df.head())



DataFrame info after datetime conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   generated_at  1 non-null      object
 1   source        1 non-null      object
 2   batches       1 non-null      object
dtypes: object(3)
memory usage: 156.0+ bytes


,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


The data is now flattened, column names are cleaned, and date/time fields are converted. The last step is to save the normalized output, typically to a CSV file.

In [ ]:
# Save the normalized output to a CSV file
output_filename = 'normalized_api_response.csv'
normalized_df.to_csv(output_filename, index=False)

print(f"Normalized data saved to '{output_filename}'")


Normalized data saved to 'normalized_api_response.csv'
